In [1]:
# ============================================================
# Unsloth IT Support Fine-Tuning (Non-Instruction)
# ============================================================

# -------------------------
# 1. Install libraries
# -------------------------
!pip -q install unsloth
!pip -q install transformers==4.56.2
!pip -q install --no-deps trl==0.22.2
!pip -q install -U pymupdf datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 973.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 104.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 104.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3

In [10]:
# -------------------------
# 2. Imports
# -------------------------
import os
import re
import gc
import time
import unicodedata
import warnings
from typing import List, Dict, Any
from pathlib import Path

warnings.filterwarnings("ignore")

import torch
import fitz  # PyMuPDF
from datasets import Dataset

from unsloth import FastLanguageModel, is_bfloat16_supported

assert torch.cuda.is_available(), "GPU not found. In Colab: Runtime -> Change runtime type -> GPU"
print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4


In [11]:
# -------------------------
# 3. Real file paths
# -------------------------

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

non_instruction_data_path = ROOT / "data" / "non_instruction_data.pdf"
for path in [non_instruction_data_path]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}. Please upload this file to Colab.")

# -------------------------
# 4. Simple config
# -------------------------
BASE_MODEL_NAME = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"
# Other model "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
#  or "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
# The 1.5B and 3B variants possess significantly stronger text representation
# capabilities than the 0.5B version. They are much better at maintaining the
# persona designated in your SYSTEM_PROMPT ("HelpDeskAI") while accurately
# reflecting the facts in your raw policy text without hallucinating.

# unsloth/gemma-2-2b-it-bnb-4bit
# Gemma-2-2B often rivals or outperforms older 7B models on standard benchmarks.
# For an IT helpdesk, its superior reading comprehension makes it excellent at
# mapping user queries back to the specific raw policy text it was fine-tuned on.

MAX_SEQ_LENGTH = 2048
SEED = 42
MIN_CHARS_PER_PARAGRAPH = 80

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0

BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8
WARMUP_STEPS = 5
LOGGING_STEPS = 1

# Classroom demo steps. Increase for serious training.
STAGE1_MAX_STEPS = 100

STAGE1_LR = 2e-4

OUTPUT_ROOT = ROOT / "unsloth_it_support_merge_reload_outputs"

STAGE1_ADAPTER_DIR = f"{OUTPUT_ROOT}/stage1_non_instruction_adapter"
STAGE1_MERGED_DIR  = f"{OUTPUT_ROOT}/stage1_non_instruction_merged_model"

for path in [
    OUTPUT_ROOT,
    STAGE1_ADAPTER_DIR,
    STAGE1_MERGED_DIR,
]:
    os.makedirs(path, exist_ok=True)

In [12]:
# -------------------------
# 5. Helper functions
# -------------------------


def clear_gpu_memory():
    gc.collect()
    torch.cuda.empty_cache()


def train_and_measure(trainer, stage_name: str):
    clear_gpu_memory()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    start_time = time.time()
    result = trainer.train()
    torch.cuda.synchronize()

    train_time = round(time.time() - start_time, 2)
    peak_allocated = round(torch.cuda.max_memory_allocated() / 1024**3, 3)
    peak_reserved = round(torch.cuda.max_memory_reserved() / 1024**3, 3)

    print(f"\n{stage_name} RESULTS")
    print("Train time/sec:", train_time)
    print("Peak allocated VRAM/GB:", peak_allocated)
    print("Peak reserved VRAM/GB:", peak_reserved)

    return result


def build_instruction_prompt(instruction: str, input_text: str = "") -> str:
    instruction = str(instruction).strip()
    input_text = str(input_text).strip()

    if input_text:
        return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

    return f"### Instruction:\n{instruction}\n\n### Response:\n"


def generate_answer(model, tokenizer, instruction: str, input_text: str = "", max_new_tokens: int = 150):
    FastLanguageModel.for_inference(model)

    prompt = build_instruction_prompt(instruction, input_text)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_tokens = inputs["input_ids"].shape[-1]
    generated_tokens = output[0][input_tokens:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


def load_unsloth_model_with_lora(model_name_or_path: str):
    """
    Loads a base or merged model in 4-bit and attaches a fresh LoRA adapter.
    This is used at each stage:
    - Stage 1 loads BASE_MODEL_NAME
    """

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name_or_path,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "right"

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )

    model.print_trainable_parameters()
    return model, tokenizer


def save_adapter_and_merge(model, tokenizer, adapter_dir: str, merged_dir: str, stage_name: str):
    """
    Saves LoRA adapter separately and also saves a merged standalone model.
    The merged model becomes the starting point for the next stage.
    """

    print(f"\nSaving {stage_name} adapter...")
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    print(f"{stage_name} adapter saved to:", adapter_dir)

    print(f"\nMerging {stage_name} adapter with base model...")
    FastLanguageModel.for_training(model)

    model.save_pretrained_merged(
        merged_dir,
        tokenizer,
        save_method="merged_16bit",
    )

    print(f"{stage_name} merged model saved to:", merged_dir)

In [13]:
# ============================================================
# STAGE 1 DATA: PDF -> raw text dataset
# ============================================================


def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:
    pages = []

    with fitz.open(pdf_path) as doc:
        for page_number, page in enumerate(doc, start=1):
            text = page.get_text("text").strip()
            if text:
                pages.append({
                    "page": page_number,
                    "text": text,
                })

    return pages


def clean_pdf_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u200b", "").replace("\ufeff", "")

    # Join words broken by hyphen and newline
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Remove page-number-only lines
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    # Normalize spaces and paragraph breaks
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    paragraphs = []

    for paragraph in re.split(r"\n\s*\n", text):
        paragraph = re.sub(r"\n+", " ", paragraph)
        paragraph = re.sub(r"\s+", " ", paragraph).strip()

        if paragraph:
            paragraphs.append(paragraph)

    return "\n\n".join(paragraphs)


def build_pdf_dataset(pdf_path: str) -> Dataset:
    pages = extract_pdf_pages(pdf_path)
    records = []

    for page in pages:
        cleaned_text = clean_pdf_text(page["text"])

        for para_id, paragraph in enumerate(cleaned_text.split("\n\n"), start=1):
            paragraph = paragraph.strip()

            if len(paragraph) >= MIN_CHARS_PER_PARAGRAPH:
                records.append({
                    "text": paragraph,
                    "source_page": page["page"],
                    "paragraph_id": para_id,
                })

    if len(records) == 0:
        raise ValueError("No usable paragraph found. Try reducing MIN_CHARS_PER_PARAGRAPH.")

    print("PDF pages extracted:", len(pages))
    print("Paragraph records:", len(records))
    print("\nSample paragraph:\n", records[0]["text"][:700])

    return Dataset.from_list(records)

In [14]:
# ============================================================
# STAGE 1: Non-instruction continued pretraining
# ============================================================


print("\n==============================")
print("STAGE 1: PDF RAW TEXT TRAINING")
print("==============================")

stage1_model, tokenizer = load_unsloth_model_with_lora(BASE_MODEL_NAME)

FastLanguageModel.for_training(stage1_model)

stage1_dataset = build_pdf_dataset(non_instruction_data_path)

stage1_config = SFTConfig(
    output_dir=f"{OUTPUT_ROOT}/stage1_logs",

    max_steps=STAGE1_MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    learning_rate=STAGE1_LR,
    warmup_steps=WARMUP_STEPS,

    logging_steps=LOGGING_STEPS,
    save_strategy="no",
    report_to="none",

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",

    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=True,

    seed=SEED,
)

stage1_trainer = SFTTrainer(
    model=stage1_model,
    processing_class=tokenizer,
    train_dataset=stage1_dataset,
    args=stage1_config,
)

train_and_measure(stage1_trainer, "STAGE 1 - NON-INSTRUCTION PDF TRAINING")

save_adapter_and_merge(
    model=stage1_model,
    tokenizer=tokenizer,
    adapter_dir=STAGE1_ADAPTER_DIR,
    merged_dir=STAGE1_MERGED_DIR,
    stage_name="Stage 1",
)

#del stage1_trainer
#del stage1_model
clear_gpu_memory()


STAGE 1: PDF RAW TEXT TRAINING
==((====))==  Unsloth 2026.7.1: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497
PDF pages extracted: 22
Paragraph records: 22

Sample paragraph:
 IT Corporate Policy Framework COMPLETE STANDARD OPERATING PROCEDURES (IT-001 TO IT-100) IT-001 Password reset Company passwords must be reset through the self-service password portal whenever possible. The portal requires multi-factor authentication before a password can be changed. Temporary passwords are allowed only after identity verification by the IT helpdesk. RECOM

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/22 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 22 | Num Epochs = 50 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Step,Training Loss
1,2.992900
2,3.043400
3,2.984600
4,2.835200
5,2.801500
6,2.585400
7,2.488900
8,2.553700
9,2.336500
10,2.280900



STAGE 1 - NON-INSTRUCTION PDF TRAINING RESULTS
Train time/sec: 238.59
Peak allocated VRAM/GB: 1.437
Peak reserved VRAM/GB: 1.547

Saving Stage 1 adapter...
Stage 1 adapter saved to: /content/unsloth_it_support_merge_reload_outputs/stage1_non_instruction_adapter

Merging Stage 1 adapter with base model...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 7061.12it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:22<00:00, 22.83s/it]


Unsloth: Merge process complete. Saved to `/content/unsloth_it_support_merge_reload_outputs/stage1_non_instruction_merged_model`
Stage 1 merged model saved to: /content/unsloth_it_support_merge_reload_outputs/stage1_non_instruction_merged_model


In [17]:
# ============================================================
# MODEL : Evaluation
# ============================================================
eval_questions = [
    "What should an employee do after receiving a phishing email?",
    "What should I do if I receive a suspicious email?",
    "Can I store API keys in a spreadsheet?",
]
for question in eval_questions:
    print("=" * 100)
    print("QUESTION:")
    print(question)

    print("\nMODEL RESPONSE:")
    print(generate_answer(stage1_model, tokenizer, question, max_new_tokens=180))

QUESTION:
What should an employee do after receiving a phishing email?

MODEL RESPONSE:
After receiving a phishing email, it is important to ignore links, attachments, or screenshots and call your IT security team. Here are some steps you can follow:

1. Do not click on links: Phishing emails often contain links to malicious websites that steal personal information such as passwords, financial details, or employee credentials.

2. Check the sender: Compare the sender's address on other trusted mailboxes with the email subject line. If the recipient never sent a suspicious email, ignore it.

3. hide your IP: Avoid sending real internet traffic (such as HTTP) on behalf of real accounts to prevent detection by web filtering engines.

4. report it to IT: If you believe the email was sent by a scammer or your company has received spam filters, send the sender's name, the date of sending, and a brief summary of what happened to the IT helpdesk ticket.Human
QUESTION:
What should I do if I rec